# The PAB database layer (Stage 1)

PAB stores all *tabular, extracted* values in a single **SQLite** database behind the thin `pab.db.store.Store` API. This notebook demonstrates create → upsert → query → export and the namespaced fit-results pivot, using an in-memory database (runs offline).

See `docs/db_schema.rst` for the full schema reference.

## Open a store

`Store.open` creates/migrates the schema, turns on foreign keys, and returns a context-managed connection.

In [1]:
from pab.db import Store, schema

store = Store.open(':memory:')
print('schema version:', schema.get_version(store.conn))
print('tables        :', list(schema.TABLE_NAMES))

schema version: 1
tables        : ['floats', 'profiles', 'mld_summary', 'granules', 'matchups', 'matchup_pixels', 'fits', 'fit_results']


## Upsert some rows

Writes go through `upsert`, keyed by each table's natural key, so the pipeline is idempotent and resumable. Here we add a float, a profile, and its mixed-layer summary.

In [2]:
store.upsert('floats', {'wmo': 6903823, 'project_name': 'BGC-Argo'})
store.upsert('profiles', {
    'wmo': 6903823, 'cycle': 387,
    'latitude': 45.0, 'longitude': -30.0,
    'time': '2024-05-01T00:00:00', 'data_mode': 'D',
})
pid = store.query('SELECT profile_id FROM profiles')[0]['profile_id']
store.upsert('mld_summary', {
    'profile_id': pid, 'mld': 42.0,
    'mld_method': 'deBoyerMontegut_0.03',
    'bbp700': 1.2e-3, 'chla': 0.35,
    'psal': 35.1, 'temp': 18.4, 'n_points': 12,
})
store.query('SELECT * FROM mld_summary')

[{'profile_id': 1,
  'mld': 42.0,
  'mld_method': 'deBoyerMontegut_0.03',
  'bbp700': 0.0012,
  'bbp700_std': None,
  'chla': 0.35,
  'chla_std': None,
  'psal': 35.1,
  'temp': 18.4,
  'n_points': 12,
  'created': None,
  'pab_version': None}]

## Idempotency

Re-upserting the same key updates the row in place — no duplicates. Re-running a pipeline stage is therefore safe.

In [3]:
store.upsert('floats', {'wmo': 6903823, 'project_name': 'updated'})
print('float rows:', store.count('floats'))
store.query('SELECT wmo, project_name FROM floats')

float rows: 1


[{'wmo': 6903823, 'project_name': 'updated'}]

## Namespaced fit results: long storage, wide view

Scalar IOP results are stored **long** (`fit_id, quantity, value, value_lo, value_hi`) so a new model pair adds rows, not columns. `fit_results_wide()` pivots to the design's namespaced columns (`BING_ExpBPow_bbp`, …) for export/reporting.

In [4]:
pv = __import__('pab').pab_version
# a matchup links a profile to a PACE granule (both must exist)
store.upsert('granules', {'granule_id': 'G1',
                          'short_name': 'PACE_OCI_L2_AOP'})
store.upsert('matchups', {
    'matchup_id': 'M1', 'profile_id': pid, 'granule_id': 'G1',
    'pab_version': pv})
store.upsert('fits', {'fit_id': 'F1', 'matchup_id': 'M1',
                      'model_pair': 'ExpBPow',
                      'pab_version': __import__('pab').pab_version})
store.upsert_many('fit_results', [
    {'fit_id': 'F1', 'quantity': 'BING_ExpBPow_bbp',
     'value': 1.5e-3, 'value_lo': 1.0e-3, 'value_hi': 2.0e-3,
     'unit': 'm^-1'},
    {'fit_id': 'F1', 'quantity': 'BING_ExpBPow_beta',
     'value': 0.8, 'value_lo': 0.6, 'value_hi': 1.0, 'unit': ''},
])
store.fit_results_wide()

quantity,BING_ExpBPow_bbp,BING_ExpBPow_beta,BING_ExpBPow_bbp_lo,BING_ExpBPow_beta_lo,BING_ExpBPow_bbp_hi,BING_ExpBPow_beta_hi
fit_id,,,,,,
F1,0.0015,0.8,0.001,0.6,0.002,1.0


## Export for inspection

The store exports any table to CSV (stdlib) or Parquet (pandas/pyarrow) for reuse in pandas.

In [5]:
import tempfile, pathlib, pandas as pd
tmp = pathlib.Path(tempfile.mkdtemp())
store.export_csv('mld_summary', tmp / 'mld.csv')
store.export_parquet('fit_results', tmp / 'fit_results.parquet')
print('CSV head:')
print((tmp / 'mld.csv').read_text().splitlines()[0])
pd.read_parquet(tmp / 'fit_results.parquet')

CSV head:
profile_id,mld,mld_method,bbp700,bbp700_std,chla,chla_std,psal,temp,n_points,created,pab_version


,fit_id,quantity,value,value_lo,value_hi,unit
0,F1,BING_ExpBPow_bbp,0.0015,0.001,0.002,m^-1
1,F1,BING_ExpBPow_beta,0.8000,0.600,1.000,


In [6]:
store.close()